In [1]:
!pip install cassandra-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cassandra-driver]


In [6]:
# docker run -d --name cassandra-server \
#  -p 9042:9042 \
#  -e CASSANDRA_CLUSTER_NAME=DemoCluster \
#  cassandra:latest
#
# docker logs -f cassandra-server

from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import uuid
from datetime import datetime

# 1. Connect to the local Docker container
# Cassandra usually has no default password unless configured
cluster = Cluster(['127.0.0.1'], port=9042)
session = cluster.connect()

def run_cassandra_demo():
    print("--- Starting Cassandra Python Demo ---")

    # 2. Create Keyspace (Think of this as a Database)
    # SimpleStrategy is fine for local demos
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS demo_keyspace 
        WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
    """)
    session.set_keyspace('demo_keyspace')

    # 3. Create Table (Schema-on-write)
    session.execute("""
        CREATE TABLE IF NOT EXISTS user_activity (
            user_id uuid,
            activity_time timestamp,
            action text,
            details text,
            PRIMARY KEY (user_id, activity_time)
        ) WITH CLUSTERING ORDER BY (activity_time DESC);
    """)

    # 4. INSERT (Write)
    user_id = uuid.uuid4()
    session.execute("""
        INSERT INTO user_activity (user_id, activity_time, action, details)
        VALUES (%s, %s, %s, %s)
    """, (user_id, datetime.now(), "LOGIN", "User logged in via Python Demo"))
    print(f"Inserted activity for user: {user_id}")

    # 5. SELECT (Read)
    rows = session.execute("SELECT * FROM user_activity LIMIT 5")
    print("\nRecent Activity Logs:")
    for row in rows:
        print(f"[{row.activity_time}] {row.action}: {row.details}")

    cluster.shutdown()
    print("\n--- Demo Complete ---")

if __name__ == "__main__":
    run_cassandra_demo()

--- Starting Cassandra Python Demo ---
Inserted activity for user: f9a0fb06-def6-4221-9262-99674a4cbdfd

Recent Activity Logs:
[2025-12-29 14:42:50.397000] LOGIN: User logged in via Python Demo

--- Demo Complete ---
